# TP 05 | Comment une mesure change-t-elle le risque de défaut ? (corrigé)

**Chapitre 4. Apprendre à partir de probabilités**

À la sortie de la presse Nervure, une alarme se déclenche. Camille doit expliquer ce que ce signal change pour le lot, comparer un modèle probabiliste aux deux finalistes du TP 04, puis transmettre une règle reproductible à l’équipe qualité.

**Parcours.** Vous passerez des effectifs aux probabilités, combinerez deux capteurs, ajusterez un classifieur et gèlerez le pipeline avec son seuil. Le livrable est un fichier local rechargeable accompagné d’une note de décision. Trois défis indépendants prolongent le parcours.

**Pour commencer.** Savoir lire un tableau et distinguer fausse alerte et défaut manqué suffit. Ouvrez le notebook dans VS Code avec le noyau Python de `.venv`. Les deux CSV du dépôt suffisent ; aucun résultat sauvegardé du TP 04 n’est nécessaire. Les exemples à la main sont fictifs. Les comparaisons de modèles utilisent ensuite les lots du même historique synthétique que les TP précédents.

## 1. Revenir des probabilités aux lots

**Atelier 5.1 du support MLB.** Sections 1 à 2.

![Mille lots répartis entre verdict et état de l’alarme](../figures/tp05-effectifs.svg)

Imaginez 1 000 lots dont 100 sont défectueux. L’alarme sonne pour 80 % des défauts et pour 20 % des lots conformes. **Avant** le signal, la probabilité de défaut $P(D)$ est le *prior*, ou probabilité a priori. $P(A\mid D)$ décrit le comportement du capteur **parmi les défauts**. Après une alarme, Camille cherche $P(D\mid A)$ : la proportion de défauts **parmi les lots signalés**. Le dénominateur a changé.

La formule de Bayes pondère chaque explication du signal par sa fréquence initiale. Avec $C$ pour « conforme », on a

$$
P(D\mid A)=
\frac{P(D)P(A\mid D)}
     {P(D)P(A\mid D)+P(C)P(A\mid C)}.
$$

Le dénominateur est la probabilité totale d’observer l’alarme. Diviser par ce total transforme deux poids en probabilités dont la somme vaut un.

### Manipulation | Que répondre à « l’alarme est fiable à 80 % » ?

![Deux groupes de référence, avec les mêmes huit carrés orange. Un carré représente dix lots.](../figures/tp05-denominateurs.svg)

Chaque carré représente dix lots. Repérez les mêmes défauts signalés dans les deux panneaux, puis comptez les carrés qui composent chacun des deux dénominateurs.

Reconstituez les quatre cases de la figure sur un brouillon. Sur 1 000 lots, combien déclenchent une alarme ? Parmi eux, quelle proportion est défectueuse ? Calculez aussi le risque en l’absence d’alarme et interprétez la différence pour Camille.

In [ ]:
# Changer explicitement de population de référence selon ce que voit Camille.
def actualiser(prior, signal_si_defaut, signal_si_conforme):
    poids_defaut = prior * signal_si_defaut
    poids_conforme = (1 - prior) * signal_si_conforme
    return poids_defaut / (poids_defaut + poids_conforme)


prior = 0.10
# Appelez actualiser pour une alarme, puis avec les probabilités complémentaires pour un silence.
apres_alarme = actualiser(prior, 0.80, 0.20)
apres_silence = actualiser(prior, 0.20, 0.80)
print(f"Alarme : {apres_alarme:.1%}. Silence : {apres_silence:.1%}.")

**Corrigé.** L’alarme rassemble 80 défauts et 180 conformes. La fraction recherchée vaut 80/260, soit 30,8 %. Parmi les 740 lots silencieux, il reste 20 défauts, soit 2,7 %. L’alarme augmente le risque initial de 10 % et le silence le réduit. Les 80 % décrivent la détection parmi les défauts. Ils ne constituent pas le risque associé à une alarme.

### Une autre inconnue : le taux de conformité d’une campagne

![Densités du taux de conformité avant et après dix contrôles](../figures/tp05-beta.svg)

Camille teste un nouveau réglage. Cette fois, l’inconnue est la proportion $\theta$ de lots conformes que le réglage produirait. La courbe décrit l’incertitude sur ce **taux**, entre 0 et 1. La probabilité d’un intervalle correspond à l’aire sous la courbe dans cet intervalle.

On utilise un prior **Beta** de paramètres positifs $a,b$. Conditionnellement à un taux stable, on suppose les verdicts indépendants et chaque lot conforme avec probabilité $\theta$. Après $s$ conformes et $f$ défauts :

$$
\begin{aligned}
\theta\mid\text{contrôles} &\sim \mathrm{Beta}(a+s,b+f)\\
E[\theta\mid\text{contrôles}] &= \frac{a+s}{a+b+s+f}.
\end{aligned}
$$

Les paramètres décrivent le poids relatif des connaissances initiales et des observations. Une série de contrôles sélectionnés uniquement sur alarme décrirait une autre population.

### Manipulation | Deux équipes interprètent huit conformes sur dix

![Évolution des deux estimations du taux de conformité quand la campagne grandit.](../figures/tp05-poids-campagne.svg)

Comparez l’écart vertical entre les deux courbes après 10 lots, puis après 100. Les campagnes gardent la même proportion de conformes.

Comparez les priors Beta(1, 1) et Beta(10, 10). Les deux équipes partent d’une moyenne de 0,5. Prévoyez laquelle changera le plus d’avis. Recommencez avec 80 conformes sur 100 et expliquez le rôle de la quantité de données.

In [ ]:
# Comparer deux convictions initiales en leur présentant exactement les mêmes contrôles.
campagnes = [(8, 2), (80, 20)]
priors_beta = [(1, 1), (10, 10)]
# Pour chaque couple, additionnez d’abord les paramètres puis calculez la moyenne.
for s, f in campagnes:
    for a, b in priors_beta:
        print((s, f), (a + s, b + f), round((a + s) / (a + b + s + f), 3))

**Corrigé.** Avec 8 conformes, on obtient Beta(9, 3), moyenne 0,750, et Beta(18, 12), moyenne 0,600. Après 100 contrôles, les moyennes passent à 0,794 et 0,750. Le prior le plus concentré garde plus de poids. À proportion observée identique, l’écart se réduit quand la campagne grandit.

### Lire deux graphes de dépendance

![Dépendances orientées entre réglage et capteurs, puis voisinages entre trois zones du panneau](../figures/tp05-dependances.svg)

À gauche, le réglage influence deux capteurs. Un réseau bayésien factorise la loi jointe en une probabilité par variable, conditionnée par ses parents. Dans ce graphe simplifié, si le réglage est connu, apprendre la chaleur ne change plus la loi de la vibration. À droite, chaque zone du panneau interagit avec sa voisine. Dans ce champ de Markov, une arête non orientée décrit une compatibilité entre états voisins. Quelle zone suffit à isoler A de C ?

Sur 100 réglages connus, on pourrait **apprendre** la fréquence de vibration pour chaque réglage. **Inférer**, c’est utiliser ces fréquences pour remonter d’une vibration observée vers le réglage probable. Si le réglage chaud est rare (10 %) et si une vibration survient dans 80 % des réglages chauds et 20 % des autres, estimez sa probabilité après une vibration.

In [ ]:
# Pondérer les deux réglages possibles par leur fréquence et la vraisemblance du signal.
poids_chaud = 0.10 * 0.80
poids_ordinaire = 0.90 * 0.20
print(
    "Probabilité de réglage chaud après vibration :",
    poids_chaud / (poids_chaud + poids_ordinaire),
)

**Corrigé.** Sur 100 réglages, on attend 8 vibrations parmi les 10 chauds et 18 parmi les 90 autres. La probabilité recherchée vaut 8/26, soit environ 31 %. Dans le champ dessiné, connaître B coupe l’unique chemin entre A et C. Sans observer B, les deux extrémités peuvent rester associées.

## 2. Combiner deux indices avec Naive Bayes

![Poids des deux classes après deux alarmes dans une campagne fictive](../figures/tp05-deux-signaux.svg)

Prenons une autre campagne : 30 % des lots y sont défectueux. Parmi les défauts, les alarmes A et B se déclenchent respectivement avec des probabilités de 0,8 et 0,6. Parmi les conformes, ces probabilités valent 0,2 et 0,4.

**Naive Bayes** suppose les entrées indépendantes **à classe fixée**. Par exemple, parmi les seuls lots défectueux, connaître A ne change alors pas la probabilité de B. Les alarmes peuvent tout de même être associées dans la population entière. Pour deux alarmes actives, on calcule un poids par classe $c$ :

$$
w_c=P(c)P(A\mid c)P(B\mid c),
\qquad P(c\mid A,B)=\frac{w_c}{\sum_k w_k}.
$$

Le dénominateur peut se lire comme un comptage. Sur 1 000 lots ayant ces proportions, on attend 300 défauts et 700 conformes. Sous l’hypothèse du modèle, les deux alarmes sonnent ensemble pour environ 144 défauts (300 × 0,8 × 0,6) et 56 conformes (700 × 0,2 × 0,4). Parmi les 200 doubles alarmes, 144 concernent donc un défaut, soit 72 %. Ce sont les mêmes lots que ceux comptés par la formule.

### Manipulation | Une deuxième alarme, puis un silence

![Trois lectures du second capteur après A, avec le multiplicateur des cotes de défaut.](../figures/tp05-branches-b.svg)

Les **cotes** sont le rapport entre probabilité de défaut et probabilité de conformité. Les flèches multiplient ces cotes par le rapport des vraisemblances. Dans cet exemple, une mesure absente n’apporte aucune information supplémentaire. Le silence observé, lui, change les cotes.

Calculez le risque après A seule, puis après A et B. Enfin, le voyant B reste éteint alors que A est allumé : quelles probabilités devez-vous remplacer par leurs compléments ? Vérifiez vos trois résultats et classez les lots par priorité.

In [ ]:
# Utiliser le résultat après A comme point de départ avant de lire le capteur B.
risque_a = actualiser(0.30, 0.80, 0.20)
# Pour B allumé, utilisez 0.60 et 0.40 ; pour B éteint, prenez leurs compléments.
risque_ab = actualiser(risque_a, 0.60, 0.40)
risque_a_sans_b = actualiser(risque_a, 0.40, 0.60)
print(risque_a, risque_ab, risque_a_sans_b)
assert abs(risque_ab - 0.72) < 1e-10

**Corrigé.** A seule porte le risque à 63,2 %. Pour deux alarmes, les poids sont 0,144 et 0,056 : leur normalisation donne 72 %. Avec A et le silence de B, les poids deviennent 0,096 et 0,084, soit 53,3 % de défauts. B silencieux est un indice observé. Une mesure B manquante ne justifierait pas de le compter comme un silence.

### Vérifier Naive Bayes en recomptant les lots

![Effectifs conservés avant et après deux alarmes](../figures/tp05-double-tri.svg)

**Nouvelle campagne fictive.** Parmi 1 000 lots, 100 sont défectueux. A s’allume sur 80 % des défauts et 20 % des conformes. B s’allume sur 60 % des défauts et 40 % des conformes. On suppose les alarmes indépendantes **à verdict fixé**. Partez des lots qui ont déjà déclenché A. Appliquez les taux de B séparément dans chaque classe. Quelle fraction des lots ayant déclenché les deux alarmes présente un défaut ? Comparez à la fraction après A seule. Que faudrait-il vérifier si les deux alarmes utilisaient le même capteur ?

In [ ]:
# Conserver les effectifs par classe pour que le dénominateur reste visible.
import numpy as np

effectifs_depart = np.array([100, 900])  # Défauts, puis conformes.
apres_a = effectifs_depart * np.array([0.80, 0.20])
taux_b = np.array([0.60, 0.40])
# Calculez les effectifs après A et B puis la part des défauts.
apres_ab = apres_a * taux_b
for etape in (apres_a, apres_ab):
    print(etape, etape[0] / etape.sum())

**Corrigé.** A retient 80 défauts et 180 conformes. Après B, il reste 48 défauts et 72 conformes. Le risque passe de 80/260, soit 30,8 %, à 48/120, soit 40 %. Si les alarmes dépendent du même capteur, mesurez les taux de B parmi les lots ayant déjà déclenché A, séparément pour chaque verdict, avant de multiplier les taux.

## 3. Passer des alarmes aux mesures continues

**Atelier 5.2 du support MLB.** Sections 3 à 5.

![Distributions gaussiennes de température ajustées sur le train par verdict](../figures/tp05-gauss.svg)

Dans les CSV, la température, la pression et la durée sont numériques. **Gaussian Naive Bayes** ajuste, pour chaque capteur et chaque classe, une courbe en cloche décrite par une moyenne et une variance. La hauteur de la courbe est une **densité**. La probabilité d’un intervalle correspond à une aire. Pour un lot mesuré en $x$, chaque capteur contribue au score de la classe $c$ :

$$
\log w_c=\log P(c)+\sum_j\log f_j(x_j\mid c).
$$

Le logarithme transforme le produit en somme et évite les problèmes numériques liés aux très petits produits. Le modèle apprend les fréquences de classe et les paramètres des densités sur le train. Une somme de contributions donne une explication du score, sans établir de lien causal entre un capteur et le défaut.

La ligne de presse est une catégorie. GaussianNB reçoit les trois mesures continues. La forêt et le boosting reprennent aussi la presse, comme au TP 04. La comparaison opérationnelle porte donc sur trois **pipelines complets**. Un défi isolera l’effet de cette différence d’entrées.

In [ ]:
# Charger les deux périodes et apprendre chaque préparation avec les seuls lots anciens.
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier

racine = Path.cwd()
if racine.name == "corrections":
    racine = racine.parent


def lire(nom):
    return pd.read_csv(
        racine / "donnees" / nom, parse_dates=["date", "label_disponible"]
    )


train, val = lire("lots_train.csv"), lire("lots_validation.csv")
assert train.label_disponible.max() < val.date.min() and set(train.id).isdisjoint(
    val.id
)
mesures = ["temperature", "pression", "duree"]
entrees = mesures + ["ligne"]


def preparer(avec_presse):
    blocs = [("mesures", SimpleImputer(strategy="median"), mesures)]
    if avec_presse:
        blocs.append(("presse", OrdinalEncoder(categories=[["A", "B"]]), ["ligne"]))
    return ColumnTransformer(blocs)


def pipeline(nom, avec_presse):
    modeles = {
        "GaussianNB": GaussianNB(),
        "Forêt": RandomForestClassifier(
            n_estimators=80,
            min_samples_leaf=5,
            max_features="sqrt",
            bootstrap=True,
            random_state=42,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=80,
            learning_rate=0.06,
            max_depth=2,
            min_child_weight=1,
            tree_method="hist",
            n_jobs=1,
            random_state=42,
        ),
    }
    return make_pipeline(preparer(avec_presse), modeles[nom])


candidats = {
    nom: pipeline(nom, nom != "GaussianNB").fit(train[entrees], train.defaut)
    for nom in ["GaussianNB", "Forêt", "XGBoost"]
}
nb = candidats["GaussianNB"]
print(len(train), "lots anciens puis", len(val), "lots de validation")
display(pd.DataFrame(nb[-1].theta_, columns=mesures, index=["conforme", "défaut"]))

### Manipulation | Quel capteur fait pencher la balance sur L0721 ?

![Calcul en cascade du log-rapport de défaut de L0721 à partir des trois mesures.](../figures/tp05-contributions.svg)

Lisez la cascade de gauche à droite. Une marche montante favorise le défaut, une descendante favorise la conformité. La dernière barre est le total à convertir en probabilité.

Retrouvez le lot et ses mesures préparées. Le tableau de calcul ci-dessous donne le log de la densité de chaque mesure sous chacune des classes. Soustrayez la ligne « conforme » de la ligne « défaut ». Une différence positive favorise le défaut. Ajoutez le logarithme du rapport des priors, puis appliquez la fonction logistique pour retrouver le score de défaut. Quel capteur fournit le plus grand indice en faveur du défaut ?

In [ ]:
# Expliquer une prédiction locale à partir des paramètres effectivement appris.
lot = val.loc[val.id.eq("L0721"), entrees]
x = nb[0].transform(lot)[0]
gauss = nb[-1]
assert gauss.classes_.tolist() == [0, 1]
log_densites = -0.5 * (
    np.log(2 * np.pi * gauss.var_) + (x - gauss.theta_) ** 2 / gauss.var_
)
display(
    pd.DataFrame(log_densites, columns=mesures, index=["conforme", "défaut"]).round(3)
)
# Convertissez le log-rapport total L en probabilité avec 1 / (1 + np.exp(-L)).
indices = pd.Series(log_densites[1] - log_densites[0], index=mesures)
log_rapport = np.log(gauss.class_prior_[1] / gauss.class_prior_[0]) + indices.sum()
score_main = 1 / (1 + np.exp(-log_rapport))
display(indices.round(3))
print("Score de défaut :", score_main)
assert np.isclose(score_main, nb.predict_proba(lot)[0, 1])

**Corrigé.** Pour L0721, pression apporte le log-rapport le plus élevé (+0.477). La somme avec le prior donne un score de défaut de 0.469. On utilise les variances du modèle, qui incluent son petit terme de stabilisation. Une contribution positive indique quelle classe explique mieux cette mesure selon les courbes apprises. Le calcul ne révèle pas le verdict.

### Deux régimes de température peuvent se recouvrir

![Affectation binaire K-means et appartenance graduelle au régime chaud](../figures/tp05-regimes.svg)

K-means affecte chaque lot au centre le plus proche. Un mélange gaussien décrit les températures par deux distributions en cloche, chacune avec son poids, sa moyenne et sa variance. L’ajustement alterne le calcul des appartenances probables et la mise à jour des composantes. Repérez sur la figure une température pour laquelle l’affectation de K-means est tranchée alors que les deux composantes du mélange restent plausibles. Dans le tableau, quelle lecture est la plus ambiguë ? Peut-on en déduire un défaut ?

In [ ]:
# Ajuster deux prototypes et deux composantes aux mêmes températures historiques.
import numpy as np
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

temperatures_valides = train.loc[
    train.temperature.between(40, 110), ["temperature"]
].to_numpy()
centres = KMeans(n_clusters=2, n_init=10, random_state=42).fit(temperatures_valides)
melange = GaussianMixture(n_components=2, n_init=5, random_state=42).fit(
    temperatures_valides
)
chaud = melange.means_[:, 0].argmax()
grille = np.linspace(*np.quantile(temperatures_valides, [0.02, 0.98]), 151)[:, None]
p_chaud = melange.predict_proba(grille)[:, chaud]
frontiere = int(np.argmin(abs(p_chaud - 0.5)))
for ligne in [0, frontiere, len(grille) - 1]:
    centre = centres.cluster_centers_[centres.predict(grille[[ligne]])[0], 0]
    print(
        f"{grille[ligne, 0]:.1f} °C | prototype {centre:.1f} °C | poids chaud {p_chaud[ligne]:.1%}"
    )

**Corrigé.** À 69,3 °C, le centre choisi est à 65,6 °C, tandis que le mélange donne 49,7 % au régime chaud. La ligne intermédiaire a un poids proche de 50 % pour chaque composante. K-means rend pourtant un seul prototype. Les deux ajustements utilisent seulement les températures. Les composantes reflètent une façon de décrire leur distribution et ne prouvent ni deux réglages physiques réels ni deux classes de qualité.

## 4. Choisir et documenter une règle

![Charge et coût selon le seuil pour les trois pipelines sur la validation](../figures/tp05-validation.svg)

Le score et le seuil remplissent deux fonctions différentes. Le score ordonne les lots selon le modèle. Le seuil inclusif transforme ce score en consigne. Appliquez le protocole du TP 04 : mêmes 240 lots, seuils de 0,05 à 1 par pas de 0,05, au plus 48 inspections. Une fausse alerte coûte 1 point, un défaut manqué en coûte 10. La sélection minimise le coût, puis la charge. À égalité, elle préfère le seuil le plus élevé puis l’ordre alphabétique du nom.

Commencez par comparer les modèles au seuil commun de 0,50. Faites ensuite le choix parmi les règles admissibles. Le test final du TP 08 ne participe ni à l’ajustement ni au choix.

In [ ]:
# Mesurer coût et charge pour chaque seuil puis classer uniquement les règles réalisables.
def evaluer(nom, scores, y):
    lignes = []
    for centieme in range(5, 101, 5):
        seuil = centieme / 100
        alerte = scores >= seuil
        vp = int((alerte & (y == 1)).sum())
        fp = int((alerte & (y == 0)).sum())
        fn = int((~alerte & (y == 1)).sum())
        lignes.append(
            {
                "modèle": nom,
                "seuil": seuil,
                "inspections": vp + fp,
                "VP": vp,
                "FP": fp,
                "FN": fn,
                "coût": fp + 10 * fn,
            }
        )
    return pd.DataFrame(lignes)


def classer(essais):
    return essais.loc[essais.inspections <= 48].sort_values(
        ["coût", "inspections", "seuil", "modèle"], ascending=[True, True, False, True]
    )


scores = {nom: m.predict_proba(val[entrees])[:, 1] for nom, m in candidats.items()}
essais = pd.concat(
    [evaluer(n, s, val.defaut.to_numpy()) for n, s in scores.items()], ignore_index=True
)
classement = classer(essais)
choix = classement.iloc[0]
display(essais.loc[essais.seuil.eq(0.5)])
display(classement.drop_duplicates("modèle"))

### À remettre | Défendre le choix devant Camille

![Décomposition du coût de chaque règle admissible en défauts manqués et fausses alertes.](../figures/tp05-prix-erreurs.svg)

Chaque barre correspond au seuil déjà réglé pour le modèle. Retrouvez sa longueur en multipliant le nombre de défauts manqués par dix puis en ajoutant les fausses alertes.

Pour chaque modèle, notez ce que le budget change par rapport au seuil de 0,50. Pour le gagnant, vérifiez les places utilisées, puis reconstituez son coût et son gain face à « aucune inspection ». Indiquez ce que les 240 lots permettent de conclure et ce qui reste à vérifier après le gel.

In [ ]:
# Contrôler la recommandation avec un scénario de référence indépendant du modèle.
cout_sans_inspection = 10 * int(val.defaut.sum())
# Retrouvez la ligne de choix dans le tableau, puis recalculez FP + 10 * FN.
print(choix.to_dict(), "gain :", cout_sans_inspection - choix["coût"])
assert choix["coût"] == choix["FP"] + 10 * choix["FN"]
assert choix["inspections"] == choix["VP"] + choix["FP"] <= 48

**Corrigé.** Au seuil de 0,50, GaussianNB demande 42 places, Forêt demande 63 places, XGBoost demande 54 places. Après réglage, XGBoost coûte 365 points au seuil 0.60, Forêt coûte 387 points au seuil 0.60, GaussianNB coûte 415 points au seuil 0.50. Le gagnant utilise 47 places pour 42 défauts interceptés. Par rapport à zéro inspection, il économise 415 points sur la validation. Ces observations ont servi à choisir le modèle. Une estimation indépendante exige des lots encore inutilisés.

## 5. Geler le pipeline et le seuil ensemble

![Du pipeline sélectionné au contrôle après rechargement](../figures/tp05-gel.svg)

**Geler** la règle consiste à conserver l’état déjà appris (imputation, encodage, modèle) et la convention de décision (classe 1 et comparaison ≥ au seuil exact). Sauvegarder seulement le classifieur ferait perdre les paramètres de préparation. Utiliser ensuite `predict` risquerait d’appliquer un autre seuil.

La cellule écrit un fichier dans `artefacts/tp05/` à la racine du projet. Son nom contient l’empreinte SHA-256 du contenu. Conservez ce nom pour le TP 08. Chaque nouvelle version dont le contenu change reçoit un autre nom. Rechargez uniquement le fichier produit localement par ce notebook : le format pickle peut exécuter du code lors de sa lecture.

![Charge de validation avec les mêmes scores XGBoost et deux seuils](../figures/tp05-deux-seuils.svg)

Imaginez que l’équipe déploie les bons poids mais saisisse 0,50 dans le service. Combien de contrôles de plus que prévu en résulteraient sur cette période ? Les sept lots supplémentaires porteraient la charge à 54, six au-delà de la capacité. Un test de réception doit donc comparer les décisions autant que les scores.

In [ ]:
# Réunir dans une seule version les paramètres appris, le seuil et la trace du choix.
import hashlib
import os
import pickle
import platform
from tempfile import NamedTemporaryFile

import sklearn
import xgboost

paquet = {
    "format": 1,
    "cours": "MLB-TP05",
    "modèle": choix["modèle"],
    "pipeline": candidats[choix["modèle"]],
    "seuil": float(choix["seuil"]),
    "comparaison": ">=",
    "classe_positive": 1,
    "colonnes": list(entrees),
    "validation": choix.to_dict(),
    "sources": {
        nom: hashlib.sha256((racine / "donnees" / nom).read_bytes()).hexdigest()
        for nom in ("lots_train.csv", "lots_validation.csv")
    },
    "versions": {
        "python": platform.python_version(),
        "sklearn": sklearn.__version__,
        "xgboost": xgboost.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
}
contenu = pickle.dumps(paquet, protocol=5)
empreinte = hashlib.sha256(contenu).hexdigest()
dossier = racine / "artefacts" / "tp05"
dossier.mkdir(parents=True, exist_ok=True)
fichier = dossier / f"regle-{empreinte}.pkl"
# Remplacer le fichier seulement après écriture complète, y compris après une relance.
with NamedTemporaryFile(dir=dossier, delete=False) as flux:
    temporaire = Path(flux.name)
    try:
        flux.write(contenu)
    except OSError:
        flux.close()
        temporaire.unlink(missing_ok=True)
        raise
try:
    os.replace(temporaire, fichier)
finally:
    temporaire.unlink(missing_ok=True)
print(fichier.relative_to(racine))

In [ ]:
# Repartir du fichier enregistré et appliquer sa convention de décision.
def ouvrir_regle(chemin):
    donnees = chemin.read_bytes()
    if chemin.stem != "regle-" + hashlib.sha256(donnees).hexdigest():
        raise ValueError("Le contenu diffère de l’empreinte indiquée dans le nom.")
    return pickle.loads(donnees)


def appliquer_seuil(regle, proba):
    if regle["comparaison"] != ">=":
        raise ValueError("Convention de seuil inconnue.")
    return proba >= regle["seuil"]


def scorer(regle, lots):
    modele = regle["pipeline"]
    colonne = list(modele.classes_).index(regle["classe_positive"])
    proba = modele.predict_proba(lots[regle["colonnes"]])[:, colonne]
    return pd.DataFrame(
        {"score": proba, "inspection": appliquer_seuil(regle, proba)}, index=lots.index
    )


regle_relue = ouvrir_regle(fichier)
rejeu = scorer(regle_relue, val)
attendu = scores[choix["modèle"]]
np.testing.assert_array_equal(rejeu.score, attendu)
np.testing.assert_array_equal(rejeu.inspection, attendu >= float(choix["seuil"]))
assert val.pression.isna().any()
display(val[["id", "pression"]].join(rejeu).head())
print(f"{len(val)} scores et décisions reproduits, y compris les lots sans pression.")

### Test de réception | Que devez-vous transmettre à l’équipe suivante ?

![Histogramme des scores rechargés et intervalle où changer le seuil change la décision.](../figures/tp05-zone-rejeu.svg)

Les lots dans la zone ombrée seraient inspectés à 0,50 et resteraient en file ordinaire avec le seuil gelé. Le bord gauche est inclus, le bord droit est exclu. Vérifiez le nombre exact dans le tableau de rejeu.

Retrouvez L0721 dans le résultat rechargé. Indiquez le nom exact du fichier, le modèle choisi, le seuil inclusif, la décision pour le lot et la taille du rejeu. Retrouvez un lot pour lequel le seuil de 0,50 et le seuil gelé conduisent à des décisions différentes, s’il en existe. Expliquez pourquoi reproduire les décisions sur les 240 lots constitue un contrôle technique et non le test final de performance.

**Vérifiez aussi la frontière.** Un collègue propose d’arrondir les scores à deux décimales avant de décider. Pour trois scores de recette situés juste sous le seuil, exactement dessus et juste au-dessus, prédisez les décisions avec et sans cet arrondi. Pourquoi la comparaison doit-elle porter sur le score complet ?

In [ ]:
# Contrôler un dossier individuel et chercher les décisions sensibles au seuil.
lecture = val[["id"]].join(rejeu)
display(lecture.loc[lecture.id.eq("L0721")])
# Comparez lecture.score.ge(0.5) à lecture.inspection et conservez les lignes différentes.
ecarts = lecture.loc[lecture.score.ge(0.5) != lecture.inspection]
display(ecarts.head(3))
print(regle_relue["modèle"], regle_relue["seuil"], fichier.name, len(ecarts), "écarts")

# Tester la frontière de la règle rechargée avec trois scores de recette.
seuil_recu = regle_relue["seuil"]
sondes = seuil_recu + np.array([-0.001, 0.0, 0.001])
exact = appliquer_seuil(regle_relue, sondes)
arrondi = appliquer_seuil(regle_relue, np.round(sondes, 2))
display(
    pd.DataFrame({"score": sondes, "décision exacte": exact, "après arrondi": arrondi})
)
assert exact.tolist() == [False, True, True]

**Corrigé.** Pour L0721, le score gelé vaut 0,712. La règle XGBoost l’applique au seuil inclusif de 0.60. Sur l’ensemble de la validation, revenir au seuil de 0,50 changerait 7 décisions. Conservez le chemin imprimé et le fichier correspondant avec la note de sélection. Les 240 scores et décisions identiques vérifient la sérialisation. Le TP 08 devra charger cette même version et l’évaluer sur la période réservée, sans nouveau fit ni nouveau choix de seuil.

Avec un seuil inclusif de 0,60, les scores 0,599, 0,600 et 0,601 donnent respectivement file ordinaire, inspection et inspection. Après arrondi, tous seraient inspectés. Arrondissez pour afficher et conservez la précision complète pour décider.

## Défi facultatif | Un taux de défaut différent sur une autre ligne

![Risque après alarme selon la fréquence initiale des défauts](../figures/tp05-prior.svg)

Le responsable annonce que la fréquence des défauts pourrait passer de 10 % à 2 % ou à 30 %. Supposez que les probabilités d’alarme conditionnelles demeurent 80 % et 20 %. Lisez les trois résultats sur la courbe, puis refaites le calcul. À partir de quel prior le risque après alarme atteint-il 50 % ?

In [ ]:
# Changer uniquement la fréquence initiale des défauts.
frequences = [0.02, 0.10, 0.30]
# Réutilisez actualiser, puis égalisez les deux poids pour trouver le point à 50 %.
for p in frequences:
    print(p, round(actualiser(p, 0.8, 0.2), 3))
assert np.isclose(actualiser(0.2, 0.8, 0.2), 0.5)

**Corrigé.** On obtient 7,5 %, 30,8 % et 63,2 %. Les deux poids sont égaux quand 0,8p = 0,2(1 − p), donc pour p = 0,20. Un même voyant peut mériter une priorité différente selon le contexte. Avant de transposer ce calcul, il faudrait vérifier que le comportement du capteur dans chaque classe reste comparable.

## Défi facultatif | Détecter un indice compté deux fois

![Une alarme copiée et deux alarmes indépendantes](../figures/tp05-copie.svg)

Un développeur du système de contrôle a dupliqué la colonne A sous un autre nom. Le second voyant vaut toujours A. Comparez le calcul naïf après deux voyants allumés au résultat correct après une seule observation. Écrivez la probabilité conjointe « A et sa copie sont allumées » pour chaque classe. Quel contrôle ajouter au dictionnaire des variables ?

In [ ]:
# Comparer une répétition informatique du même signal à deux observations indépendantes.
p_un = actualiser(0.30, 0.8, 0.2)
# Réappliquez les mêmes vraisemblances comme si la copie apportait un nouvel indice.
p_double = actualiser(p_un, 0.8, 0.2)
print(f"Une information : {p_un:.1%}. Calcul naïf avec copie : {p_double:.1%}.")

**Corrigé.** Le risque correct reste 63,2 %. Le calcul naïf le porte à 87,3 %. Si le second voyant copie A, la probabilité conjointe est 0,8 parmi les défauts et 0,2 parmi les conformes, exactement comme pour A seule. Examiner la provenance des champs et rechercher les copies dans chaque classe évite cette fausse confirmation.

## Défi facultatif | Comparer à capteurs identiques

![Coûts de validation avec les entrées du parcours principal et avec trois capteurs identiques.](../figures/tp05-memes-entrees.svg)

Pour GaussianNB, les symboles se superposent : il utilisait déjà les trois capteurs. Pour les ensembles, chaque symbole correspond au meilleur seuil admissible de la variante concernée.

Reprenez la figure coût-charge. Pour séparer le choix de l’algorithme de l’apport de la presse, réentraînez la forêt et le boosting avec les trois mesures uniquement. Conservez les hyperparamètres, la grille de seuils et la contrainte de 48 places. Quelle comparaison pouvez-vous désormais défendre ? Utilisez des objets distincts pour conserver la règle gelée.

In [ ]:
# Construire une comparaison parallèle sans modifier le pipeline de la sélection principale.
comparables = [essais.loc[essais["modèle"].eq("GaussianNB")]]
# Utilisez pipeline(nom, False) pour entraîner les deux ensembles sur les trois mesures.
# Calculez les scores puis appelez evaluer et classer sur le tableau combiné.
for nom in ["Forêt", "XGBoost"]:
    variante = pipeline(nom, False).fit(train[entrees], train.defaut)
    comparables.append(
        evaluer(nom, variante.predict_proba(val[entrees])[:, 1], val.defaut.to_numpy())
    )
comparaison = classer(pd.concat(comparables)).drop_duplicates("modèle")
display(comparaison)

**Corrigé.** À capteurs identiques, les meilleurs coûts admissibles sont 375 pour XGBoost, 396 pour Forêt, 415 pour GaussianNB. Comparez ces valeurs aux finalistes avec la presse. Cette expérience teste les algorithmes à entrées communes. Elle ne permet pas d’isoler une cause physique. Le fichier gelé garde la décision du parcours principal.

### Suite du parcours

Conservez le pipeline et le seuil pour le TP 08. Dans le [TP 06](../TP06_derive_de_ligne.ipynb), Camille suit la presse au cours du temps. Son nouvel indicateur déclenche une vérification de la ligne. La règle gelée ici continue à servir à la sélection des lots.

### Pour approfondir

[Naive Bayes dans scikit-learn](https://scikit-learn.org/stable/modules/naive_bayes.html) | [Sauvegarde et rechargement des modèles](https://scikit-learn.org/stable/model_persistence.html)